In [3]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import matplotlib.pyplot as plt
import pandas as pd
import os
import random
from collections import deque

class MahimahiTraceManager:
    """
    Mengelola file trace Mahimahi (.log).
    Mengonversi timestamp (ms) menjadi Throughput (Mbps) dengan amplifikasi yang realistis.
    """
    def __init__(self, folder_path="traces_folder"):
        self.traces = []
        PACKET_SIZE_BITS = 1500 * 8 
        
        if os.path.exists(folder_path):
            files = [f for f in os.listdir(folder_path) if f.endswith('.log')]
            files.sort()
            
            for file in files:
                path = os.path.join(folder_path, file)
                try:
                    with open(path, 'r') as f:
                        timestamps_ms = [float(line.strip()) for line in f if line.strip()]
                        
                        if timestamps_ms:
                            throughput_mbps = []
                            current_sec = 0
                            packet_count = 0
                            
                            for ts in timestamps_ms:
                                sec = int(ts / 1000)
                                while current_sec < sec:
                                    mbps = (packet_count * PACKET_SIZE_BITS) / 1_000_000
                                    throughput_mbps.append(mbps)
                                    packet_count = 0
                                    current_sec += 1
                                packet_count += 1
                            
                            throughput_mbps.append((packet_count * PACKET_SIZE_BITS) / 1_000_000)
                            
                            # Scaling Realistis (Memaksa Konflik)
                            max_tp = max(throughput_mbps) if throughput_mbps else 1
                            scale_factor = 12.0 / max_tp if max_tp > 0 else 1.0
                            
                            scaled_mbps = [max(0.1, tp * scale_factor) for tp in throughput_mbps]
                            
                            # Smoothing agar tidak terlalu banyak noise spike (window 3 detik)
                            smoothed = pd.Series(scaled_mbps).rolling(window=3, min_periods=1).mean().tolist()
                            
                            self.traces.append({
                                "name": file,
                                "data": smoothed
                            })
                except Exception as e:
                    print(f"Gagal memproses file {file}: {e}")
        
        if not self.traces:
            print("⚠️ Folder traces_folder tidak ditemukan atau kosong!")
            self.traces.append({"name": "synth_fallback", "data": [8.0] * 500})

        self.active_trace = None
        self.ptr = 0

    def select_random_trace(self):
        self.active_trace = random.choice(self.traces)
        self.ptr = random.randint(0, max(0, len(self.active_trace["data"]) - 110))
        return self.active_trace["name"]

    def set_trace_index(self, idx):
        self.active_trace = self.traces[idx % len(self.traces)]
        self.ptr = 0
        return self.active_trace["name"]

    def get_next_bandwidth(self):
        val = self.active_trace["data"][self.ptr]
        self.ptr = (self.ptr + 1) % len(self.active_trace["data"])
        return val

class FluidStreamingEnv(gym.Env):
    def __init__(self, trace_manager):
        super(FluidStreamingEnv, self).__init__()
        self.trace_manager = trace_manager
        self.bitrates = [0.5, 2.5, 8.0] # Kualitas 0, 1, 2
        
        # REVISI STATE: [Buffer, Current_TP, Avg_TP_3_Steps, LastAction, BufferTrend, TPTrend]
        # Ukuran observation_space dinaikkan menjadi 6 elemen.
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0, 0.0, -1.0, -1.0]),
            high=np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0]),
            dtype=np.float32
        )
        self.action_space = spaces.Discrete(3)
        self.state = None 
        self.max_steps = 100
        self.current_step = 0
        self.tp_history = deque(maxlen=3) # Memori historis jaringan

    def _get_normalized_obs(self):
        """Mengonversi raw state menjadi angka ternormalisasi untuk jaringan PPO"""
        buffer, current_tp, avg_tp, last_action, buf_trend, tp_trend = self.state
        
        norm_buffer = np.clip(buffer / 30.0, 0.0, 1.0)           
        norm_current_tp = np.clip(current_tp / 15.0, 0.0, 1.0)               
        norm_avg_tp = np.clip(avg_tp / 15.0, 0.0, 1.0)               
        norm_action = float(last_action) / 2.0                   
        norm_buf_trend = np.clip(buf_trend / 10.0, -1.0, 1.0)    
        
        # Mengurangi bobot tp_trend agar tidak panik (pembagi diubah jadi 20.0)
        norm_tp_trend = np.clip(tp_trend / 20.0, -1.0, 1.0)      
        
        return np.array([norm_buffer, norm_current_tp, norm_avg_tp, norm_action, norm_buf_trend, norm_tp_trend], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if options and "trace_idx" in options:
            trace_name = self.trace_manager.set_trace_index(options["trace_idx"])
        else:
            trace_name = self.trace_manager.select_random_trace()
            
        initial_tp = self.trace_manager.get_next_bandwidth()
        
        # Inisialisasi history throughput
        self.tp_history.clear()
        for _ in range(3):
            self.tp_history.append(initial_tp)
            
        # State Asli (Raw): [Buffer, Current_TP, Avg_TP, Action, Buf_Trend, TP_Trend]
        self.state = np.array([10.0, initial_tp, initial_tp, 1.0, 0.0, 0.0], dtype=np.float32)
        self.current_step = 0
        return self._get_normalized_obs(), {"trace": trace_name}

    def step(self, action):
        if isinstance(action, np.ndarray):
            action = action.item()
        
        buffer, _, last_avg_tp, last_action, _, _ = self.state
        chosen_bitrate = self.bitrates[int(action)]
        raw_tp = self.trace_manager.get_next_bandwidth()
        
        # Update history dan hitung rata-rata throughput
        self.tp_history.append(raw_tp)
        avg_tp = sum(self.tp_history) / len(self.tp_history)
        
        seg_duration = 5.0
        download_time = (chosen_bitrate * seg_duration / (raw_tp + 0.1))
        stalling = max(0, download_time - buffer)
        
        new_buffer = max(0, buffer - download_time) + seg_duration
        new_buffer = min(new_buffer, 30.0)
        
        buf_trend = new_buffer - buffer
        # Tren TP sekarang dihitung berdasarkan rata-rata, bukan lonjakan titik sesaat
        tp_trend = avg_tp - last_avg_tp
        
        # --- REVISI REWARD STABILITAS JANGKA PANJANG ---
        reward = float(chosen_bitrate * 1.0) 
        
        if stalling > 0:
            reward -= float(stalling * 20.0) 
            reward -= 5.0 
            
        # Penalti Switching diperbesar (Mikir 2x sebelum ganti kualitas)
        reward -= float(abs(int(action) - int(last_action)) * 2.5)

        # Bonus Stabilitas: Diberikan jika agen konsisten dan tidak menyebabkan macet
        if int(action) == int(last_action) and stalling == 0:
            reward += 1.0

        self.state = np.array([new_buffer, raw_tp, avg_tp, float(action), buf_trend, tp_trend], dtype=np.float32)
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        info = {
            "raw_tp": raw_tp, 
            "buffer": new_buffer,
            "actual_bitrate": chosen_bitrate
        }
        
        return self._get_normalized_obs(), reward, done, False, info

def run_experiment():
    log_dir = "./rl_logs/"
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    
    tm = MahimahiTraceManager(folder_path="traces_folder/mahimahi")
    if not tm.traces:
        print("❌ Tidak ada file trace yang ditemukan.")
        return
        
    env = Monitor(FluidStreamingEnv(tm), log_dir)
    
    print("⏳ Memulai pelatihan agen adaptif (300,000 langkah)...")
    model = PPO("MlpPolicy", env, verbose=1, 
                learning_rate=0.0002,  
                ent_coef=0.08,         
                n_steps=2048,
                batch_size=64)         
    
    model.learn(total_timesteps=300000)
    model.save("fluid_ndn_ai_model")
    print("✅ Pelatihan selesai. Model disimpan.")

    print("\n📈 Menghasilkan 8 Grafik Evaluasi...")
    for i in range(len(tm.traces)):
        obs, info = env.reset(options={"trace_idx": i})
        trace_name = info["trace"]
        history = []
        
        for _ in range(100):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info_step = env.step(action)
            action_val = action.item() if isinstance(action, np.ndarray) else int(action)
            
            history.append({
                'Throughput': info_step['raw_tp'],
                'Buffer': info_step['buffer'], 
                'Action': action_val,
                'Bitrate': info_step['actual_bitrate']
            })
            
        df = pd.DataFrame(history)
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        
        ax1.plot(df.index, df['Throughput'], label='Bandwidth Trace (Mbps)', color='blue', alpha=0.3)
        ax1.step(df.index, df['Bitrate'], label='Keputusan Bitrate AI (Mbps)', color='red', linewidth=2.5, where='post')
        ax1.set_title(f"Evaluasi Trace Dinamis: {trace_name}")
        ax1.set_ylabel("Mbps")
        ax1.legend()
        ax1.grid(alpha=0.2)
        
        ax2.fill_between(df.index, df['Buffer'], color='green', alpha=0.15, label='Level Isi Buffer')
        ax2.axhline(y=10, color='orange', linestyle='--', label='Garis Bahaya (10s)')
        ax2.set_ylabel("Detik")
        ax2.set_xlabel("Nomor Segmen")
        ax2.set_ylim(0, 35)
        ax2.legend()
        ax2.grid(alpha=0.2)
        
        plt.tight_layout()
        plot_filename = os.path.join(log_dir, f"result_{trace_name}.png")
        plt.savefig(plot_filename)
        plt.close()

    try:
        x, y = ts2xy(load_results(log_dir), 'timesteps')
        plt.figure(figsize=(10, 5))
        plt.plot(x, pd.Series(y).rolling(window=100).mean(), color='purple')
        plt.title("Track Record Kemajuan Pelatihan (Reward)")
        plt.xlabel("Total Langkah")
        plt.ylabel("Rata-rata Skor")
        plt.savefig(os.path.join(log_dir, "training_progress.png"))
        plt.close()
    except Exception as e:
        print(f"⚠️ Gagal menyimpan progress pelatihan: {e}")

if __name__ == "__main__":
    run_experiment()

/root/.pyenv/versions/vevn/lib/python3.11/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/root/.pyenv/versions/vevn/lib/python3.11/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


⏳ Memulai pelatihan agen adaptif (300,000 langkah)...
Using cpu device
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 100       |
|    ep_rew_mean     | -3.07e+03 |
| time/              |           |
|    fps             | 4771      |
|    iterations      | 1         |
|    time_elapsed    | 0         |
|    total_timesteps | 2048      |
----------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 100           |
|    ep_rew_mean          | -2.96e+03     |
| time/                   |               |
|    fps                  | 3430          |
|    iterations           | 2             |
|    time_elapsed         | 1             |
|    total_timesteps      | 4096          |
| train/                  |               |
|    approx_kl            | 0.00039683495 |
|    clip_fraction        | 0             |
|    clip_range 